In [ ]:
from pathlib import Path
import random
import pickle
from typing import Literal

import pandas as pd
import numpy as np
from joblib import Parallel, delayed

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(42)

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)

real = pd.concat(dfs, ignore_index=True)

In [ ]:
version = "2025-03-28 15:06:33"
fname = f"synthetic_dfs_n_samples=100_patientflow_{version}_seed=42.pkl"

with open(fname, "rb") as f:
    gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
def rule_one_onset_rule(gen_df: pd.DataFrame):
    correct_patients = np.ones(len(gen_df["REF"].unique()), dtype=np.bool8)

    not_related_or_unknown = ["not related", "unknown"]

    for pid, (_, pdf) in enumerate(gen_df.groupby(by="REF")):
        row = pdf.reset_index(drop=True).iloc[0]
        if row["Onset"] in ["bulbar", "axial/resp"]:
            if (
                row["Limb_O"].lower() in not_related_or_unknown
                and row["Limbs_Impairment"].lower() in not_related_or_unknown
                and row["Limbs_Side"].lower() in not_related_or_unknown
            ):
                correct_patients[pid] = True
            else:
                correct_patients[pid] = False

        elif row["Onset"] == "limbs":
            if (
                row["Limb_O"].lower() != "not related"
                and row["Limbs_Impairment"].lower() != "not related"
                and row["Limbs_Side"].lower() != "not related"
            ):
                correct_patients[pid] = True
            else:
                correct_patients[pid] = False

    return correct_patients

In [ ]:
def rule_two_alsfrsr_progression(
    gen_df: pd.DataFrame,
    delta=2,
    delta_type: Literal["historic_min", "from_last"] = "from_last",
):
    correct_patients = np.ones(len(gen_df["REF"].unique()), dtype=np.bool8)

    for pid, (_, pdf) in enumerate(gen_df.groupby(by="REF")):
        for q_id in range(1, 13):
            pdf_q = pdf[f"P{q_id}"].reset_index(drop=True)
            if delta_type == "from_last":
                for i in range(1, len(pdf_q)):
                    if pdf_q.iloc[i - 1] < pdf_q.iloc[i] - delta:
                        correct_patients[pid] = False
                        break
            elif delta_type == "historic_min":
                historical_min = pdf_q.iloc[0]
                for i in range(1, len(pdf_q)):
                    if pdf_q.iloc[i] > historical_min + delta:
                        correct_patients[pid] = False
                        break
                    historical_min = min(historical_min, pdf_q.iloc[i])
    return correct_patients

In [ ]:
def check_rules(gen_df: pd.DataFrame):
    rules = [
        rule_one_onset_rule(gen_df),
        rule_two_alsfrsr_progression(gen_df, delta=1, delta_type="historic_min"),
    ]

    return rules

In [ ]:
parallel = Parallel(n_jobs=-1, verbose=10)
rule_checks = np.array(parallel(delayed(check_rules)(gen_df) for gen_df in gen_dfs))

In [ ]:
real_rule_one = rule_one_onset_rule(real)

In [ ]:
np.invert(real_rule_one).sum()

In [ ]:
real_rule_two = rule_two_alsfrsr_progression(real, delta=1, delta_type="historic_min")

In [ ]:
real_rule_two_breaking_patients = np.invert(real_rule_two).sum()
real_rule_two_breaking_patients

In [ ]:
for rule_id in range(0, 2):
    print(f"Rule {rule_id + 1}")
    if rule_id == 1:
        passing_datasets = (
            np.invert(rule_checks)[:, rule_id].sum(axis=1)
            <= real_rule_two_breaking_patients
        ).sum()
    else:
        passing_datasets = rule_checks[:, rule_id].all(axis=1).sum()
    print(f"Passing Datasets: {passing_datasets}")
    mean_not_passing = (np.invert(rule_checks))[:, rule_id].sum(axis=1).mean()
    std_not_passing = (np.invert(rule_checks))[:, rule_id].sum(axis=1).std()
    print(
        f"Mean and avg number of patients not passing the rule: {mean_not_passing} +/- {std_not_passing}\n"
    )

In [ ]:
with open(
    f"rule_checks_historical_min_n_samples=100_patientflow_{version}_seed=42.pkl", "wb"
) as f:
    pickle.dump(rule_checks, f)